# Data Prep from Silver to Gold

Requires silver parquet data

In gold layer we add features (especial spatial features like hexagon) and aggregate data from different datasets, etc.

In [ ]:
import pandas as pd
import duckdb
import polars as pl
import numpy as np
import geopandas as gpd
from shapely import wkt
import h3
from datetime import datetime
import math


SILVER_TAXI_PATH = "../data/processed_data/silver_taxi.parquet"
SILVER_WEATHER_PATH = "../data/processed_data/silver_weatherdata.parquet"
SILVER_HEXAGON_PATH = "../data/processed_data/silver_dim_h3_chicago_8.parquet"

GOLD_TAXI_PATH = "../data/processed_data/gold_taxi.parquet"
GOLD_WEATHER_PATH = "../data/processed_data/gold_weather.parquet"
GOLD_HOURLY_DEMAND = "../data/processed_data/gold_hourly_demand.parquet"

# Hexagon related 
H3_RESOLUTION = 8

# Weather related
START_TS = "2024-01-01 00:00:00"
END_TS = "2026-05-01 23:00:00" # also max date in taxi data

Nachdem in silver alle Duplikate bereinigt wurden, können nun trip_id und taxi_id gedropped werden

# Taxi Data

## Adding hexagon

First we add hexagon data and then compare the hexagons of trips against the city boundaries of chicago. We only want trips with pickup hexagon in the city boundaries

In [4]:
taxi_with_h3 = (
    pl.scan_parquet(SILVER_TAXI_PATH)
    .with_columns(
        pl.struct(["pickup_centroid_latitude", "pickup_centroid_longitude"])
        .map_elements(
            lambda row: h3.latlng_to_cell(
                row["pickup_centroid_latitude"],
                row["pickup_centroid_longitude"],
                H3_RESOLUTION,
            ),
            return_dtype=pl.String,
        )
        .alias("pickup_h3_cell"),
        
        pl.struct(["dropoff_centroid_latitude", "dropoff_centroid_longitude"])
        .map_elements(
            lambda row: h3.latlng_to_cell(
                row["dropoff_centroid_latitude"],
                row["dropoff_centroid_longitude"],
                H3_RESOLUTION,
            ),
            return_dtype=pl.String,
        )
        .alias("dropoff_h3_cell"),
    )
)

# Check if there are trips without hexagon in Chicagos boundaries
valid_chicago_h3_cells = (
    pl.scan_parquet(SILVER_HEXAGON_PATH)
    .select("h3_cell")
)

count_before = (
    taxi_with_h3
    .select(pl.len().alias("n_rows_before"))
    .collect()
)

taxi_with_h3_chicago_only = (
    taxi_with_h3
    .join(
        valid_chicago_h3_cells,
        left_on="pickup_h3_cell",
        right_on="h3_cell",
        how="inner",
    )
)

count_after = (
    taxi_with_h3_chicago_only
    .select(pl.len().alias("n_rows_after"))
    .collect()
)

print(count_before)
print(count_after)

/var/folders/sw/7cvjbzt5623085bdv7hpg6vw0000gn/T/ipykernel_89677/1267103421.py:53: UserWarning: Extension type 'geoarrow.wkb' is not registered; loading as its storage type.

To avoid this warning, register the extension type or set environment variable 'POLARS_UNKNOWN_EXTENSION_TYPE_BEHAVIOR' to 'load_as_storage' or 'load_as_extension'.

In Polars 2.0, the default behavior will change to 'load_as_extension'.
  .collect()


shape: (1, 1)
┌───────────────┐
│ n_rows_before │
│ ---           │
│ u32           │
╞═══════════════╡
│ 13389077      │
└───────────────┘
shape: (1, 1)
┌──────────────┐
│ n_rows_after │
│ ---          │
│ u32          │
╞══════════════╡
│ 13387586     │
└──────────────┘


## Save gold version


In [5]:
taxi_gold.sink_parquet(GOLD_TAXI_PATH)

print(f"Silver taxi parquet written to: {GOLD_TAXI_PATH}")

Silver taxi parquet written to: ../data/processed_data/gold‚_taxi.parquet


# Weather Data

In [11]:
def mode_or_na(series: pd.Series):
    """Return most frequent non-null value, otherwise NA."""
    mode_values = series.dropna().mode()
    if len(mode_values) == 0:
        return pd.NA
    return mode_values.iloc[0]


def create_hourly_weather_gold(
    df: pd.DataFrame,
    start_ts: str = START_TS,
    end_ts: str = END_TS,
) -> pd.DataFrame:
    """
    Create hourly gold weather dataset.

    Steps:
    - Parse valid timestamp
    - Restrict to requested date range
    - Aggregate observations to hourly level
    - Create complete hourly timestamp spine
    - Reindex to all hours
    - Linearly interpolate numeric weather columns
    - Fill categorical/context columns
    - Add date/hour helper columns
    """

    df = df.copy()

    # 1. Basic cleanup
    df["valid"] = pd.to_datetime(df["valid"], utc=True)
    df = df.dropna(subset=["valid"])
    
    # Convert UTC to chicago timezone
    df["valid"] = df["valid"].dt.tz_convert("America/Chicago")

    start_ts = (
        pd.Timestamp(start_ts)
        .tz_localize(
            "America/Chicago",
            ambiguous=False, # takes winter time hour
            nonexistent="shift_forward"
        )
    )

    end_ts = (
        pd.Timestamp(end_ts)
        .tz_localize(
            "America/Chicago",
            ambiguous=False, # takes winter time hour
            nonexistent="shift_forward"
        )
    )

    # Optional: keep only relevant date range
    df = df[(df["valid"] >= start_ts) & (df["valid"] <= end_ts)]

    # 2. Create hourly timestamp
    df["valid_hour"] = df["valid"].dt.floor("h", ambiguous=False, nonexistent="shift_forward")

    # 3. Cast numeric columns
    numeric_cols = [
        "tmpc",   # temperature Celsius
        "relh",   # relative humidity
        "sknt",   # wind speed in knots
        "p01m",   # precipitation
        "vsby",   # visibility
        "lat",
        "lon",
    ]

    existing_numeric_cols = [col for col in numeric_cols if col in df.columns]

    for col in existing_numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 4. Define aggregation rules
    agg_dict = {}

    # Numeric weather values: hourly mean
    for col in ["tmpc", "relh", "sknt", "vsby"]:
        if col in df.columns:
            agg_dict[col] = "mean"

    # Precipitation: hourly sum is usually more sensible than mean
    if "p01m" in df.columns:
        agg_dict["p01m"] = "sum"

    # Station/location fields
    if "station" in df.columns:
        agg_dict["station"] = mode_or_na

    if "lat" in df.columns:
        agg_dict["lat"] = "mean"

    if "lon" in df.columns:
        agg_dict["lon"] = "mean"

    # Categorical weather condition
    if "skyc1" in df.columns:
        agg_dict["skyc1"] = mode_or_na

    # 5. Aggregate to hourly level
    hourly = (
        df
        .groupby("valid_hour", as_index=True)
        .agg(agg_dict)
        .sort_index()
    )

    # 6. Complete hourly index from start to end
    full_hourly_index = pd.date_range(
        start=start_ts,
        end=end_ts,
        freq="h",
        name="valid_hour",
    )

    hourly = hourly.reindex(full_hourly_index)

    # 7. Interpolate numeric columns over missing hours
    numeric_interpolate_cols = [
        col for col in ["tmpc", "relh", "sknt", "p01m", "vsby", "lat", "lon"]
        if col in hourly.columns
    ]

    hourly[numeric_interpolate_cols] = (
        hourly[numeric_interpolate_cols]
        .interpolate(method="time", limit_direction="both")
    )

    # 8. Fill categorical/context columns
    categorical_fill_cols = [
        col for col in ["station", "skyc1"]
        if col in hourly.columns
    ]

    for col in categorical_fill_cols:
        hourly[col] = hourly[col].ffill().bfill()

    # 9. Back to normal dataframe
    gold = hourly.reset_index()

    # 10. Add helper columns for joins / ML
    # TODO might drop some later if not used for joining
    gold["date"] = gold["valid_hour"].dt.date
    gold["year"] = gold["valid_hour"].dt.year
    gold["month"] = gold["valid_hour"].dt.month
    gold["day"] = gold["valid_hour"].dt.day
    gold["hour"] = gold["valid_hour"].dt.hour
    gold["weekday"] = gold["valid_hour"].dt.weekday

    # 11. Optional quality flags
    original_hours = set(df["valid_hour"].dropna().unique())

    gold["was_observed_hour"] = gold["valid_hour"].isin(original_hours)
    gold["was_interpolated_hour"] = ~gold["was_observed_hour"]
    
    # Onehot encoding
    skyc1_dummies = pd.get_dummies(
        gold["skyc1"],
        prefix="skyc1",
        dummy_na=False,
        dtype="int8",
    )

    gold = pd.concat([gold, skyc1_dummies], axis=1)
    
    # Drop unused columns
    gold = gold.drop(columns=["skyc1", "station", "lat", "lon"])

    return gold

weather_silver = pd.read_parquet(SILVER_WEATHER_PATH)

weather_gold = create_hourly_weather_gold(
    weather_silver,
    start_ts=START_TS,
    end_ts=END_TS,
)

weather_gold.to_parquet(
    GOLD_WEATHER_PATH,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

print(f"Gold weather parquet written to: {GOLD_WEATHER_PATH}")
print(f"Rows: {len(weather_gold):,}")
print(f"Start: {weather_gold['valid_hour'].min()}")
print(f"End: {weather_gold['valid_hour'].max()}")
print()
print(weather_gold.head().to_string(index=False))
print()
print(weather_gold.tail().to_string(index=False))

Gold weather parquet written to: ../data/processed_data/gold_weather.parquet
Rows: 20,447
Start: 2024-01-01 00:00:00-06:00
End: 2026-05-01 23:00:00-05:00

               valid_hour  tmpc  relh  sknt  vsby  p01m       date  year  month  day  hour  weekday  was_observed_hour  was_interpolated_hour  skyc1_BKN  skyc1_CLR  skyc1_FEW  skyc1_OVC  skyc1_SCT  skyc1_VV 
2024-01-01 00:00:00-06:00  1.11 75.26  12.0  10.0   0.0 2024-01-01  2024      1    1     0        0               True                  False          0          0          0          1          0          0
2024-01-01 01:00:00-06:00  1.11 75.26  12.0   8.0   0.0 2024-01-01  2024      1    1     1        0               True                  False          0          0          0          1          0          0
2024-01-01 02:00:00-06:00  0.56 81.63  10.0   7.0   0.0 2024-01-01  2024      1    1     2        0               True                  False          0          0          0          1          0          0
2024-01-01 03

In [12]:
duckdb.sql(f"""
    SELECT hour, count(*)
    FROM  read_parquet('{GOLD_WEATHER_PATH}')
    GROUP BY hour
    ORDER BY hour asc       
           """).show(max_rows=100)

duckdb.sql(f"""
    SELECT date, hour, was_interpolated_hour
    FROM  read_parquet('{GOLD_WEATHER_PATH}')
    WHERE was_interpolated_hour = True    
           """).show(max_rows=100)

┌───────┬──────────────┐
│ hour  │ count_star() │
│ int32 │    int64     │
├───────┼──────────────┤
│     0 │          852 │
│     1 │          854 │
│     2 │          849 │
│     3 │          852 │
│     4 │          852 │
│     5 │          852 │
│     6 │          852 │
│     7 │          852 │
│     8 │          852 │
│     9 │          852 │
│    10 │          852 │
│    11 │          852 │
│    12 │          852 │
│    13 │          852 │
│    14 │          852 │
│    15 │          852 │
│    16 │          852 │
│    17 │          852 │
│    18 │          852 │
│    19 │          852 │
│    20 │          852 │
│    21 │          852 │
│    22 │          852 │
│    23 │          852 │
└───────┴──────────────┘
  24 rows    2 columns

┌────────────┬───────┬───────────────────────┐
│    date    │ hour  │ was_interpolated_hour │
│    date    │ int32 │        boolean        │
├────────────┼───────┼───────────────────────┤
│ 2024-11-03 │     1 │ true                  │
│ 2025-03-09 │  

# Merged Dataset

Here we create an hourly dataset, whereby each hour contains a row for each hexagon. 

In [4]:
# First we need the "blueprint" dataset which contains each hour and hexagon combination from start to end date
START_DATETIME = datetime.strptime(
    START_TS,
    "%Y-%m-%d %H:%M:%S",
)
END_DATETIME = datetime.strptime(
    END_TS,
    "%Y-%m-%d %H:%M:%S",
)

hours = pl.DataFrame({
    "datetime_hour": pl.datetime_range(
        start=START_DATETIME,
        end=END_DATETIME,
        interval="1h",
        closed="both",
        eager=True,
    )
})

hexagons = (
    pl.scan_parquet(SILVER_HEXAGON_PATH)
    .select([
        "h3_cell"
    ])
    .unique()
    .collect()
)

hexagon_hourly_grid = (
    hours
    .join(
        hexagons,
        how="cross",
    )
)

n_hours = hours.height
n_hexagons = hexagons.height
expected_rows = n_hours * n_hexagons

print(f"Hours:          {n_hours:,}")
print(f"Hexagons:       {n_hexagons:,}")
print(f"Expected rows:  {expected_rows:,}")
print(f"Actual rows:    {hexagon_hourly_grid.height:,}")

# hexagon_hourly_grid.write_parquet(GOLD_HOURLY_DEMAND)

Hours:          20,448
Hexagons:       853
Expected rows:  17,442,144
Actual rows:    17,442,144


Features creation

We use cyclic encoding for columns like hour, weekday or month to be aware of the distance between time instances (e. g. hour 23 -> 0)

In [ ]:
hexagon_hourly_features = (
    hexagon_hourly_grid
    .with_columns([
        pl.col("datetime_hour").dt.month().alias("month"),
        pl.col("datetime_hour").dt.weekday().alias("weekday"),
        pl.col("datetime_hour").dt.hour().alias("hour"),
    ])
    .with_columns([
        (pl.col("weekday") >= 6).alias("is_weekend"),

        # month: 1-12 -> 0-11
        ((2 * math.pi * (pl.col("month") - 1) / 12).sin()).alias("month_sin"),
        ((2 * math.pi * (pl.col("month") - 1) / 12).cos()).alias("month_cos"),

        # weekday: 1-7 -> 0-6
        ((2 * math.pi * (pl.col("weekday") - 1) / 7).sin()).alias("weekday_sin"),
        ((2 * math.pi * (pl.col("weekday") - 1) / 7).cos()).alias("weekday_cos"),

        # hour: already 0-23
        ((2 * math.pi * pl.col("hour") / 24).sin()).alias("hour_sin"),
        ((2 * math.pi * pl.col("hour") / 24).cos()).alias("hour_cos"),
    ])
)

Join the new dataset with the hourly weather data

In [ ]:
weather_hourly = (
    pl.scan_parquet(GOLD_WEATHER_PATH)
    .with_columns(
        pl.col("valid_hour")
        .dt.replace_time_zone(None)
        .dt.truncate("1h")
        .alias("datetime_hour")
    )
)

hexagon_hourly_with_weather = (
    hexagon_hourly_features
    .join(
        weather_hourly,
        on="datetime_hour",
        how="left",
    )
)

hexagon_hourly_with_weather.sink_parquet(GOLD_HOURLY_DEMAND)